# colab_08 — Integration of balanced 100k + 100k subsamples

Integrates **Bhaduri 2020 organoids (100k)** and **Bhaduri 2021 fetal cortex (100k)** into a shared embedding using Harmony batch correction.

Replaces colab_03, which integrated Bhaduri 2020 (224k) + Zhong 2018 (2,330) at a 100× imbalance. The downstream DPT pathology in Session 15 traced back to that imbalance plus disconnected graph components from rare Zhong-only cell types (microglia, OPCs, RBC). Sessions 16–20 replaced Zhong with Bhaduri 2021 and stratified-subsampled both datasets to 100k.

## Inputs
- `data/processed/bhaduri_2020/bhaduri_2020_100k.h5ad` — 100,000 cells × 16,774 genes, `.raw` set (counts in `.raw.X`)
- `data/processed/bhaduri_2021/bhaduri_2021_100k.h5ad` — 100,000 cells × 33,694 genes, `raw=False` (counts in `.X`)

## Output
- `data/processed/integrated/integrated_100k_harmony.h5ad`

## Notable differences from colab_03
1. No Zhong-specific dense-to-sparse fix (2c in colab_03) — doesn't apply.
2. Asymmetric raw handling: 2020 uses `.raw.X`; 2021's `.X` is already counts.
3. No off-target cluster removal pre-integration (option A): 2021's Outlier + `cell_type=='0'` cells already filtered in colab_07; 2020 stress/choroid clusters are kept and labeled in colab_09 (post-integration).
4. Gene intersection: 16,768 (vs 14,498 with Zhong).

## 0. Setup

### 0a — Mount Drive and import packages

Mounts Google Drive (project lives at `MyDrive/brain-organoid-trajectories`), imports the standard scanpy stack plus harmonypy. Defines `PATHS` for both inputs and the integration output.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import scipy.sparse as sp
import harmonypy as hm
import matplotlib.pyplot as plt

sc.settings.verbosity = 1
sc.settings.set_figure_params(dpi=80, frameon=False, figsize=(6, 5))

DRIVE_ROOT = '/content/drive/MyDrive/brain-organoid-trajectories'
PATHS = {
    'bhaduri_2020_100k': os.path.join(DRIVE_ROOT, 'data/processed/bhaduri_2020/bhaduri_2020_100k.h5ad'),
    'bhaduri_2021_100k': os.path.join(DRIVE_ROOT, 'data/processed/bhaduri_2021/bhaduri_2021_100k.h5ad'),
    'integrated_out':    os.path.join(DRIVE_ROOT, 'data/processed/integrated/integrated_100k_harmony.h5ad'),
}
os.makedirs(os.path.dirname(PATHS['integrated_out']), exist_ok=True)
print('scanpy', sc.__version__, '| anndata', ad.__version__, '| harmonypy', hm.__version__)

## 1. Load both 100k subsamples

### 1a — Read both h5ads from Drive

Loads each AnnData and prints shape, raw status, and obs columns. The two datasets store counts differently — we unify in section 2.

In [ ]:
adata_2020 = sc.read_h5ad(PATHS['bhaduri_2020_100k'])
adata_2021 = sc.read_h5ad(PATHS['bhaduri_2021_100k'])

print(f'Bhaduri 2020: {adata_2020.shape[0]:,} cells x {adata_2020.shape[1]:,} genes  '
      f'(raw is {"set" if adata_2020.raw is not None else "None"})')
print(f'Bhaduri 2021: {adata_2021.shape[0]:,} cells x {adata_2021.shape[1]:,} genes  '
      f'(raw is {"set" if adata_2021.raw is not None else "None"})')
print()
print('Bhaduri 2020 obs cols:', list(adata_2020.obs.columns))
print('Bhaduri 2021 obs cols:', list(adata_2021.obs.columns))

## 2. Unify count storage — both `.X` should hold raw integer counts

The two subsamples store counts asymmetrically:
- **Bhaduri 2020**: `.raw` is set (raw counts in `.raw.X`), `.X` holds scaled values from colab_02.
- **Bhaduri 2021**: `raw=False`, raw counts already in `.X`.

Old colab_03 did `adata.X = adata.raw.X.copy()` for both inputs, which would crash on 2021 (`.raw is None`). We branch instead, then sanity-check that both `.X` matrices are sparse non-negative integers.

### 2a — Move raw counts into `.X` (asymmetric handling)

For 2020: copy `.raw.X` into `.X`. For 2021: leave `.X` alone. Ensure both are CSR-sparse afterwards.

In [ ]:
# 2020: replace .X with raw counts from .raw.X
if adata_2020.raw is None:
    raise ValueError('Bhaduri 2020 .raw is unexpectedly None — re-check colab_07 output')
adata_2020.X = adata_2020.raw.X.copy()

# 2021: .X is already counts (raw=False from colab_07). Warn if .raw was set.
if adata_2021.raw is not None:
    print('WARN: Bhaduri 2021 .raw unexpectedly set — still using .X (assumed counts)')

# ensure both are sparse CSR
for name, a in [('2020', adata_2020), ('2021', adata_2021)]:
    if not sp.issparse(a.X):
        a.X = sp.csr_matrix(a.X)
    elif a.X.format != 'csr':
        a.X = a.X.tocsr()
    print(f'Bhaduri {name}: dtype={a.X.dtype}, format={a.X.format}, nnz={a.X.nnz:,}')

### 2b — Sanity check that `.X` is non-negative integer counts

Samples up to 1000 nonzero entries per matrix. Counts should be non-negative integers. If we see floats with fractional parts or negative values, something pre-normalized leaked through.

In [ ]:
for name, a in [('2020', adata_2020), ('2021', adata_2021)]:
    n = min(1000, a.X.nnz)
    sample = a.X.data[:n]
    is_int_like = np.allclose(sample, sample.astype(int))
    print(f'Bhaduri {name}: min={sample.min()}, max={sample.max()}, '
          f'dtype={sample.dtype}, integer-like={is_int_like}')

## 3. Shared gene space

### 3a — Intersect `var_names` across datasets

Integration requires identical gene panels. Expected from Session 20 sanity check: ca. 16,768 shared genes — the 6 Bhaduri-2020-only genes are `.1`-suffixed Cell Ranger duplicates and biologically irrelevant. Subset both datasets to the shared genes in sorted order.

In [ ]:
shared_genes = adata_2020.var_names.intersection(adata_2021.var_names).sort_values()

print(f'Bhaduri 2020 genes: {adata_2020.shape[1]:,}')
print(f'Bhaduri 2021 genes: {adata_2021.shape[1]:,}')
print(f'Shared:             {len(shared_genes):,}')
print(f'Lost from 2020:     {adata_2020.shape[1] - len(shared_genes):,}')
print(f'Lost from 2021:     {adata_2021.shape[1] - len(shared_genes):,}')

adata_2020 = adata_2020[:, shared_genes].copy()
adata_2021 = adata_2021[:, shared_genes].copy()

print()
print(f'After subset — 2020: {adata_2020.shape}')
print(f'After subset — 2021: {adata_2021.shape}')

## 4. Concatenate with dataset label

### 4a — Prefix barcodes and concat

Prefix obs names with `b2020_` / `b2021_` so combined barcodes are globally unique. `ad.concat(..., label='dataset')` adds an obs column named `dataset` with values `bhaduri_2020` / `bhaduri_2021` — this becomes Harmony's batch key.

Dataset-specific obs columns (e.g. `protocol`, `age_week`, `cell_type_coarse`, `age_gw`, `donor`, `area_ucsc`) are not in both objects, so they will be `NaN`-filled or dropped depending on `merge` strategy. We re-attach what we need post-integration in colab_09.

In [ ]:
adata_2020.obs_names = 'b2020_' + adata_2020.obs_names
adata_2021.obs_names = 'b2021_' + adata_2021.obs_names
print('First 3 barcodes 2020:', adata_2020.obs_names[:3].tolist())
print('First 3 barcodes 2021:', adata_2021.obs_names[:3].tolist())

adata = ad.concat(
    {'bhaduri_2020': adata_2020, 'bhaduri_2021': adata_2021},
    axis=0,
    join='outer',
    label='dataset',
    merge='same',
    index_unique=None,
)
adata.obs['dataset'] = adata.obs['dataset'].astype('category')

# free per-dataset copies
del adata_2020, adata_2021

print()
print(f'Combined: {adata.shape[0]:,} cells x {adata.shape[1]:,} genes')
print(adata.obs['dataset'].value_counts())

## 5. Normalize, log-transform, select HVGs

### 5a — Save raw counts to layer, then CP10K-normalize and log-transform

Stash counts in `layers['counts']` so HVG selection (5b) can use the raw-count `seurat_v3` flavor while `.X` carries log(CP10K + 1) for downstream PCA.

In [ ]:
adata.layers['counts'] = adata.X.copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)
print(f'.X after log1p: dtype={adata.X.dtype}, mean={adata.X.mean():.3f}, max={adata.X.max():.2f}')

### 5b — Select 2,000 HVGs (seurat_v3, per-batch)

`flavor='seurat_v3'` reads raw counts from `layer='counts'` and ranks genes by normalized variance, computed per `batch_key='dataset'` and merged. Picks 2,000 genes consistently variable across both datasets. Save `.raw = adata` (full log-normalized gene space) before subsetting `.X` to HVGs — marker plots later use `use_raw=True` to access the full panel.

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=2000,
    flavor='seurat_v3',
    layer='counts',
    batch_key='dataset',
)
adata.raw = adata
adata = adata[:, adata.var['highly_variable']].copy()
print(f'HVGs selected: {adata.shape[1]:,} genes')
print(f'Highly variable in N batches: '
      f'{adata.var["highly_variable_nbatches"].value_counts().sort_index().to_dict()}')

## 6. Scale and PCA

### 6a — Scale (clip max=10) and run 30-component PCA

Matches Session 11's choice of 30 PCs for the unbalanced run. Note: `sc.pp.scale` densifies `.X` (HVG × cells), expected ca. 1.5 GB at 200k × 2000 × 4 bytes.

In [ ]:
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, n_comps=30, svd_solver='arpack', random_state=42)
print(f'X_pca shape: {adata.obsm["X_pca"].shape}')
print(f'Variance ratio (PC1–5): {np.round(adata.uns["pca"]["variance_ratio"][:5], 4)}')
print(f'Cumulative variance through PC30: {adata.uns["pca"]["variance_ratio"].sum():.3f}')

## 7. Harmony batch correction

### 7a — Run Harmony directly (workaround for sc.external bug)

`sc.external.pp.harmony_integrate` has a shape bug in current scanpy/harmonypy versions (Session 11). We call `hm.run_harmony` directly. The memory's note from colab_03 was "assign `ho.Z_corr` without transpose" — we add a shape assertion to catch any version-driven flip.

In [ ]:
ho = hm.run_harmony(
    adata.obsm['X_pca'],
    meta_data=adata.obs,
    vars_use=['dataset'],
    max_iter_harmony=50,
    random_state=42,
)
Z = ho.Z_corr
if Z.shape != adata.obsm['X_pca'].shape:
    print(f'Note: Z_corr shape {Z.shape} != X_pca shape {adata.obsm["X_pca"].shape} — transposing')
    Z = Z.T
assert Z.shape == adata.obsm['X_pca'].shape, f'Harmony output shape mismatch: {Z.shape}'
adata.obsm['X_pca_harmony'] = Z
print(f'X_pca_harmony shape: {adata.obsm["X_pca_harmony"].shape}')

## 8. Neighbor graph, UMAP, Leiden

### 8a — Compute neighbor graph on Harmony embedding

`use_rep='X_pca_harmony'` so neighbors reflect batch-corrected distances. `n_neighbors=15` is scanpy default.

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, use_rep='X_pca_harmony', random_state=42)
print('Neighbor graph computed on X_pca_harmony.')

### 8b — UMAP

2D embedding for inspection. Random state fixed for reproducibility.

In [ ]:
sc.tl.umap(adata, random_state=42)
print(f'X_umap shape: {adata.obsm["X_umap"].shape}')

### 8c — Leiden clustering at resolution 0.5

Matches Session 11. Expected count: ca. 15–25 clusters.

In [ ]:
sc.tl.leiden(adata, resolution=0.5, random_state=42)
n_clusters = adata.obs['leiden'].nunique()
print(f'Leiden clusters: {n_clusters}')
print(adata.obs['leiden'].value_counts().sort_index())

## 9. Visual diagnostics

### 9a — UMAP colored by `dataset` (mixing check)

If Harmony worked, the two colors should intermix within each cell-type region. Stripes / dataset-segregated regions = Harmony failed or under-corrected.

In [ ]:
sc.pl.umap(adata, color='dataset', frameon=False, save='_dataset.png', show=True)

### 9b — UMAP colored by Leiden cluster

Cluster geography. We assign cell types in colab_09 (next notebook).

In [ ]:
sc.pl.umap(adata, color='leiden', legend_loc='on data', frameon=False, save='_leiden.png', show=True)

### 9c — Lineage marker genes

Sanity-check that markers land in expected regions:
- **SOX2**, **PAX6** — radial glia
- **EOMES** — intermediate progenitors
- **TBR1**, **NEUROD2** — excitatory neurons
- **GAD1**, **GAD2** — GABAergic interneurons
- **GFAP** — astrocytes
- **MKI67** — cycling cells

Scattered/diffuse markers → over-integration. `use_raw=True` reads from the full 16,768-gene `.raw` matrix, not the 2,000-HVG `.X`.

In [ ]:
markers = ['SOX2', 'PAX6', 'EOMES', 'TBR1', 'NEUROD2', 'GAD1', 'GAD2', 'GFAP', 'MKI67']
present = [g for g in markers if g in adata.raw.var_names]
missing = [g for g in markers if g not in adata.raw.var_names]
if missing:
    print(f'Markers missing from .raw: {missing}')
sc.pl.umap(adata, color=present, use_raw=True, ncols=3, frameon=False, save='_markers.png', show=True)

### 9d — Cluster × dataset composition

Percentage of each dataset within each Leiden cluster. Values near 50/50 mean Harmony mixed that cluster well; clusters > 95% one dataset are candidates for batch artifacts — unlike the colab_03 Zhong run, both Bhaduri datasets should largely overlap, so heavily skewed clusters here would be more concerning than in colab_03.

In [ ]:
comp = pd.crosstab(adata.obs['leiden'], adata.obs['dataset'], normalize='index') * 100
comp = comp.round(1)
comp['total_cells'] = adata.obs['leiden'].value_counts().sort_index().values
print(comp)

## 10. Save integrated object

### 10a — Write `integrated_100k_harmony.h5ad` to Drive

Keeps `.raw` (full log-normalized 16,768-gene matrix), `.X` (scaled HVGs), `.obsm['X_pca']` / `X_pca_harmony` / `X_umap`, neighbor graph, and Leiden labels. Predicted size: ca. 6–7 GB (extrapolated from colab_03's 226k → 7.5 GB at the same HVG/PC settings).

In [ ]:
adata.write_h5ad(PATHS['integrated_out'])
size_gb = os.path.getsize(PATHS['integrated_out']) / 1e9
print(f'Saved: {PATHS["integrated_out"]}')
print(f'Size:  {size_gb:.2f} GB')
print(f'Final shape: {adata.shape}')